# Chapter 3: Multiple Regression Analysis — Estimation

Topics: Multiple OLS in practice, OLS in matrix form, omitted variable bias, ceteris paribus interpretation.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 3.1 Multiple Regression in Practice

## Step 2 – The Multiple Regression Model

**Population model** with k regressors:
$$y_i = \beta_0 + \beta_1 x_{1i} + \beta_2 x_{2i} + \cdots + \beta_k x_{ki} + u_i$$

**Matrix notation:**
$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \mathbf{u}$$

where $\mathbf{y}$ is $(n\times 1)$, $\mathbf{X}$ is $(n\times(k+1))$ (first column all ones), $\boldsymbol{\beta}$ is $((k+1)\times 1)$.

**OLS estimator** (normal equations):
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{y}$$

Key assumption: $\text{rank}(\mathbf{X}) = k+1$ (no perfect multicollinearity).

In [ ]:
df = woo.dataWoo('wage1')

# Simple vs multiple regression
m1 = smf.ols('wage ~ educ', data=df).fit()
m2 = smf.ols('wage ~ educ + exper', data=df).fit()
m3 = smf.ols('wage ~ educ + exper + tenure', data=df).fit()

print("Simple regression (educ only):")
print(f"  β_educ = {m1.params['educ']:.4f}")
print("\nMultiple regression (educ + exper):")
print(f"  β_educ = {m2.params['educ']:.4f}")
print("\nFull model (educ + exper + tenure):")
print(m3.summary())

## 3.2 OLS in Matrix Form

## Step 3 – OLS in Matrix Form (Manual Calculation)

**Step-by-step matrix OLS:**

1. Build design matrix $\mathbf{X}$ (add column of ones for intercept)
2. Compute $\mathbf{X}'\mathbf{X}$ — a $(k+1)\times(k+1)$ symmetric matrix
3. Invert: $(\mathbf{X}'\mathbf{X})^{-1}$ using `np.linalg.inv()`
4. Compute $\mathbf{X}'\mathbf{y}$
5. Multiply: $\hat{\boldsymbol{\beta}} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{y}$

**Variance-covariance matrix of $\hat{\boldsymbol{\beta}}$:**
$$\text{Var}(\hat{\boldsymbol{\beta}}) = \sigma^2(\mathbf{X}'\mathbf{X})^{-1}$$

Estimated using $\hat{\sigma}^2 = SSR/(n-k-1)$. Standard errors are square roots of diagonal elements.

In [ ]:
# Manual OLS via normal equations: β = (X'X)^{-1} X'y
X = np.column_stack([np.ones(len(df)), df['educ'], df['exper'], df['tenure']])
y = df['wage'].values

XtX_inv = np.linalg.inv(X.T @ X)
beta_hat = XtX_inv @ X.T @ y
print("OLS coefficients (matrix form):")
for name, val in zip(['Intercept','educ','exper','tenure'], beta_hat):
    print(f"  {name}: {val:.4f}")

# Verify against statsmodels
print("\nStatsmodels coefficients:")
print(m3.params.round(4))

## 3.3 Omitted Variable Bias

## Step 4 – Omitted Variable Bias

If the true model is $y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + u$ but we omit $x_2$, the short regression estimator is biased:

$$E[\tilde{\beta}_1] = \beta_1 + \beta_2 \cdot \delta_1$$

where $\delta_1 = \frac{\widehat{\text{Cov}}(x_1, x_2)}{\widehat{\text{Var}}(x_1)}$ is the slope from regressing $x_2$ on $x_1$.

**Direction of bias:**

| $\beta_2 > 0$ | $\delta_1 > 0$ | Bias: positive (upward) |
|---|---|---|
| $\beta_2 > 0$ | $\delta_1 < 0$ | Bias: negative (downward) |
| $\beta_2 < 0$ | $\delta_1 > 0$ | Bias: negative (downward) |
| $\beta_2 < 0$ | $\delta_1 < 0$ | Bias: positive (upward) |

In [ ]:
# Omitting 'exper' from wage regression
m_full    = smf.ols('wage ~ educ + exper + tenure', data=df).fit()
m_omitted = smf.ols('wage ~ educ + tenure', data=df).fit()

print("With exper:   β_educ =", round(m_full.params['educ'], 4))
print("Without exper: β_educ =", round(m_omitted.params['educ'], 4))

# Correlation between omitted var and included vars
print(f"\nCorr(educ, exper) = {df['educ'].corr(df['exper']):.4f}")
print("Omitted var bias direction: negative (exper negatively correlated with educ, positively with wage)")

## Comparing Models: R² across specifications

In [ ]:
models = {
    'educ only': m1,
    'educ + exper': m2,
    'educ + exper + tenure': m3
}

fig, ax = plt.subplots(figsize=(8, 4))
names = list(models.keys())
r2    = [m.rsquared for m in models.values()]
ax.bar(names, r2, color=['#1f77b4','#ff7f0e','#2ca02c'])
ax.set_ylabel('R²')
ax.set_title('R² Across Model Specifications')
ax.set_ylim(0, max(r2)*1.3)
for i, v in enumerate(r2):
    ax.text(i, v + 0.002, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_r2_comparison.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_03_Multiple_Regression_Estimation_01.png'), dpi=150, bbox_inches='tight')
plt.show()